# Чистовик: константный прогноз

Заготовка, которая уже проходит весь путь: читает `data/`, считает CV по фолдам, пишет `submission.csv` и `cv.json` и печатает `cv.json` последней строкой. Замените прогноз и разбиение на свои.

In [60]:
import json, sys
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

In [61]:
sys.path.insert(0, "data")
from metric_cv import quantile_quality, cv_json

tr = pd.read_csv("data/train.csv")
sample = pd.read_csv("data/sample_submission.csv")
y = tr["daily_orders"].to_numpy(float)
QUANTILES = (0.1, 0.5, 0.9)

In [ ]:
def predict(y_train, n):
    """Константный прогноз: квантили спроса по всему train. Замените на модель."""
    
    q = np.quantile(y_train, QUANTILES)
    print(q)
    return q
    


In [63]:
# CV: четыре случайных фолда — замените разбиением, которое похоже на тест
splits = list(KFold(4, shuffle=True, random_state=0).split(tr))
oof = np.zeros((len(tr), 3))
for a, b in splits:
    oof[b] = predict(y[a], len(b))
folds = [100 * quantile_quality(y[b], oof[b, 0], oof[b, 1], oof[b, 2], tr["city"].iloc[b]) for _, b in splits]

In [80]:
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

tr = pd.read_csv("data/train.csv")


X = tr.drop(columns=['daily_orders','id'])
y = tr['daily_orders']


kf = KFold(n_splits=5,shuffle=False)
kf.get_n_splits()
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()





te = pd.read_csv("data/test.csv")


y_te = np.zeros(len(te))

for i, (train_index, test_index) in enumerate(kf.split(X)):
    cb = CatBoostRegressor(early_stopping_rounds=50,cat_features=categorical_cols,verbose=False)
    cb.fit(X.iloc[train_index],y.iloc[train_index])
    y_pred = cb.predict(X.iloc[test_index])

    y_te += cb.predict(te[X.columns])


    rmse = root_mean_squared_error(y.iloc[test_index],y_pred)
    print(f"Fold {i}:")
    print(f"  Test:     {rmse}")

y_te = y_te/5
    


/tmp/ipykernel_179651/161843839.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()


Fold 0:
  Test:     33.062061487770336
Fold 1:
  Test:     32.86259998118738
Fold 2:
  Test:     29.84118344113127
Fold 3:
  Test:     29.268365466917196
Fold 4:
  Test:     28.705205761007733


In [81]:
# посылка: колонки и строки как в sample_submission.csv, 0 <= p10 <= p50 <= p90 <= 5000

test = predict(y_te, len(te))
sub = pd.DataFrame({"id": te["id"],
                    "predicted_daily_orders_p10": test[0],
                    "predicted_daily_orders_p50": test[1],
                    "predicted_daily_orders_p90": test[2]})
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample)
assert set(sub["id"]) == set(sample["id"]) and (sub.iloc[:, 1:].to_numpy() >= 0).all()
sub.to_csv("submission.csv", index=False)

cv = cv_json("constant", [f / 100 for f in folds])   # пишет cv.json
print(json.dumps(cv))                                  # последняя строка — для проверки воспроизводимости


[ 25.32057274  57.54401474 242.33691461]
{"name": "constant", "metric": "quantile_quality_x100", "folds": [0.0, 0.0, 0.0, 0.0], "cv": 0.0, "std": 0.0}


In [84]:
import json
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import KFold

tr = pd.read_csv("data/train.csv")
te = pd.read_csv("data/test.csv")

X = tr.drop(columns=["daily_orders", "id"])
y = tr["daily_orders"]
te_filtered = te[X.columns]

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

# 2. Настройка KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

y_te_p10 = np.zeros(len(te_filtered))
y_te_p50 = np.zeros(len(te_filtered))
y_te_p90 = np.zeros(len(te_filtered))

folds_rmse = []  

for i, (train_index, test_index) in enumerate(kf.split(X)):
    X_train, y_train = X.iloc[train_index], y.iloc[train_index]
    X_val, y_val = X.iloc[test_index], y.iloc[test_index]

    # Модель для p10 (Квантиль 0.10)
    cb_p10 = CatBoostRegressor(
        loss_function="Quantile:alpha=0.10",
        early_stopping_rounds=50,
        cat_features=categorical_cols,
        verbose=False,
    )
    cb_p10.fit(X_train, y_train, eval_set=(X_val, y_val))
    y_te_p10 += cb_p10.predict(te_filtered)

    # Модель для p50 (Квантиль 0.50 / Медиана)
    cb_p50 = CatBoostRegressor(
        loss_function="Quantile:alpha=0.50",
        early_stopping_rounds=50,
        cat_features=categorical_cols,
        verbose=False,
    )
    cb_p50.fit(X_train, y_train, eval_set=(X_val, y_val))
    y_te_p50 += cb_p50.predict(te_filtered)

    y_val_pred = cb_p50.predict(X_val)
    fold_rmse = root_mean_squared_error(y_val, y_val_pred)
    folds_rmse.append(fold_rmse)
    print(f"Fold {i}: Validation p50 RMSE = {fold_rmse:.4f}")

    cb_p90 = CatBoostRegressor(
        loss_function="Quantile:alpha=0.90",
        early_stopping_rounds=50,
        cat_features=categorical_cols,
        verbose=False,
    )
    cb_p90.fit(X_train, y_train, eval_set=(X_val, y_val))
    y_te_p90 += cb_p90.predict(te_filtered)

y_te_p10 /= 5
y_te_p50 /= 5
y_te_p90 /= 5



sub = pd.DataFrame(
    {
        "id": te["id"],
        "predicted_daily_orders_p10": y_te_p10,
        "predicted_daily_orders_p50": y_te_p50,
        "predicted_daily_orders_p90": y_te_p90,
    }
)



sub.to_csv("submission.csv", index=False)
print("\nSubmission saved successfully! Sample rows:")
print(sub.head())

# 7. Запись cv.json (используем ваши переменные)
# Предполагается, что ваша функция cv_json принимает имя и список скоров
# cv = cv_json("catboost_quantile", folds_rmse)
# print(json.dumps(cv))


/tmp/ipykernel_179651/2890850312.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()


Fold 0: Validation p50 RMSE = 30.6325
Fold 1: Validation p50 RMSE = 36.2112
Fold 2: Validation p50 RMSE = 31.2286
Fold 3: Validation p50 RMSE = 30.3521
Fold 4: Validation p50 RMSE = 28.4961

Submission saved successfully! Sample rows:
              id  predicted_daily_orders_p10  predicted_daily_orders_p50  \
0  PVZ_TE_000001                   29.037565                   34.072668   
1  PVZ_TE_000002                  151.413685                  178.838615   
2  PVZ_TE_000003                   25.035336                   26.800539   
3  PVZ_TE_000004                   43.988384                   57.957437   
4  PVZ_TE_000005                  112.793347                  138.160758   

   predicted_daily_orders_p90  
0                   46.024714  
1                  246.744855  
2                   32.597901  
3                   74.746282  
4                  184.390125  
